# b04의 접힘점·소켓 추정점·YOLO 골반 비교
**두 장에 같은 한 점을 찍습니다.** 선수 본인의 왼쪽에서 **몸통 앞면과 허벅지 윗면이 만나 안으로 가장 깊게 꺾이는 점**을 고르세요. 팔이나 원판이 가리면 ‘가려서 판독 불가’, 어느 위치인지 확신이 없으면 ‘위치 불확실’을 선택하세요.

**모든 셀 실행(Run All Cells)** 후 아래 탭에서 9번과 21번을 각각 판독합니다. 클릭할 때마다 자동 저장됩니다. 두 장을 끝내고 **비교 결과 보기**를 누르세요. 그때 처음으로 예측과 기존 소켓 추정점이 표시됩니다.

### 1. 기존 분할과 원본 확인

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

from wlcv.annotation.pose_points import HIP_FOLD, pose_point_editor
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS
from wlcv.evaluation.hip_fold import FRAMES, METHODS, VIDEO, compare, fold_reads, plot_comparison

info = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").loc[VIDEO]
role = pd.read_csv(SPLIT).set_index("video_id").at[VIDEO, "role"]
assert (info.person_id, role, info.near_side) == ("b04", "train", "left")
video_path = VIDEOS / info.filename
fold_folder = ANNOTATIONS / "train_hip_fold"
socket_folder = ANNOTATIONS / "train_hip_review"
output = ROOT / "outputs/pose_baseline/train_hip_fold"

### 2. 접힘점 저장 후 두 기준 비교
9번은 이륙 후보, 21번은 무릎 통과입니다. 원본 크기로 확대해 확인할 수 있습니다. 소켓은 프레임마다 기존 두 수동 판독의 중점을 사용합니다. 어깨는 해당 프레임의 YOLO 왼쪽 어깨 예측 하나를 세 조건에 공통으로 사용합니다.

결과표의 거리는 화면에서 YOLO 골반과 각 수동 기준 사이의 직선거리입니다. 각도는 어깨에서 각 끝점으로 이은 선의 수직선 기준 기울기입니다. 변화량은 21번 각도에서 9번 각도를 뺍니다. 음수이면 세워진 방향, 양수이면 더 기울어진 방향입니다. 좌표가 없거나 모델 관절 신뢰도가 0.50 미만이면 해당 계산을 비워 둡니다. 작은 변화의 부호는 참고용입니다.

In [ ]:
editors = []
for frame in FRAMES:
    editors.append(pose_point_editor(
        video_path, frame, info.person_id, info.near_side,
        split=role, annotations=fold_folder, joints=(HIP_FOLD,),
    ))
tabs = widgets.Tab(children=editors)
for index, frame in enumerate(FRAMES):
    tabs.set_title(index, f"b04 · {frame}번")

button = widgets.Button(description="비교 결과 보기")
report = widgets.Output()

def show_results(_):
    report.clear_output(wait=True)
    with report:
        labels = fold_reads(fold_folder)
        pending = [frame for frame, point in labels.items() if point is None]
        if pending:
            print(f"먼저 {pending}번의 접힘점 또는 판독 불가 상태를 저장하세요.")
            return
        weights = ROOT / "models/yolo26n-pose.pt"
        if not weights.is_file():
            print(f"로컬 자세 추정 가중치를 확인하세요: {weights}")
            return
        points = compare(video_path, YOLO(str(weights)), fold_folder, socket_folder)
        for frame, point in labels.items():
            if point["status"] == "unreadable":
                print(f"{frame}번 접힘점은 판독 불가입니다. 해당 거리와 각도는 비워 둡니다.")
        angles = points.pivot(index="기준", columns="프레임", values="각도 (°)").reindex(METHODS)
        angles["변화 (°)"] = angles[21] - angles[9]
        display(points[["프레임", "기준", "YOLO와 거리 (px)", "어깨 신뢰도", "골반 신뢰도"]].round(2))
        display(angles.round(2))
        fig = plot_comparison(video_path, points)
        output.mkdir(parents=True, exist_ok=True)
        points.to_csv(output / "frames.csv", index=False)
        angles.to_csv(output / "angle_changes.csv")
        fig.savefig(output / "hip_fold.png", dpi=140, bbox_inches="tight")
        plt.show()

button.on_click(show_results)
display(widgets.VBox([tabs, button, report]))